# Step 7: Deep baseline, trained from scratch
**CSC 481 · Sidra Azeem**

A convolutional network that reads the number **straight from the torso crop**, with no hand-built steps.
It is **designed and trained here from scratch** (no pretrained weights, as the course requires), and it
replaces the earlier ImageNet-pretrained ResNet-18.

* **Three outputs** (as in Grad 2025 and Golovkin et al. 2025): tens digit (0–9 or "none"), ones digit (0–9),
  and "is a number visible?".
* **Same players and frames** as the classical notebook: same dev / val split, same 6 val and test frames per
  player, same player-level vote and metrics. So the results compare directly.
* Training from scratch needs more data, so the network trains on **16 frames per dev player** (the
  classical pipeline used 6).

**Runtime → Change runtime type → T4 GPU.** About 20–30 minutes in total.

## Setup

In [ ]:
FRAMES_TRAIN = 16          # frames per dev player used for training
EPOCHS = 30

import os, glob, json, random, math, re, time, zipfile
from collections import Counter, defaultdict
import numpy as np, cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '(switch to a T4 GPU runtime if this says cpu)')

from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/csc481_soccer'
ZIPS = os.path.join(WORK, 'SoccerNet', 'jersey-2023')
LOCAL = '/content/jersey'                      # 6 frames per player (same as the classical notebook)
LOCAL_TRAIN = f'/content/jersey{FRAMES_TRAIN}'  # more frames per player, for training only

## Data
The split code is copied from the classical notebook, so dev / val / test are **exactly the same players**.

In [ ]:
def extract_subset(zip_path, out_dir, k):
    """Extract the label file + k evenly spaced frames per player."""
    with zipfile.ZipFile(zip_path) as z:
        by_t, keep = defaultdict(list), []
        for n in z.namelist():
            m = re.search(r'images/([^/]+)/[^/]+\.jpg$', n)
            if m: by_t[m.group(1)].append(n)
            elif n.endswith('_gt.json'): keep.append(n)
        for files in by_t.values():
            files.sort()
            keep += [files[i] for i in np.linspace(0, len(files) - 1, min(k, len(files))).round().astype(int)]
        for n in keep:
            z.extract(n, out_dir)
    print(f'{os.path.basename(zip_path)} -> {out_dir}: {len(by_t)} players, {len(keep)} files')

for split in ['train', 'test']:
    if not os.path.exists(os.path.join(LOCAL, split)):
        extract_subset(os.path.join(ZIPS, f'{split}.zip'), LOCAL, 6)
if not os.path.exists(os.path.join(LOCAL_TRAIN, 'train')):
    extract_subset(os.path.join(ZIPS, 'train.zip'), LOCAL_TRAIN, FRAMES_TRAIN)

def load_split(root, split):
    gts = glob.glob(os.path.join(root, '**', f'{split}_gt.json'), recursive=True)
    labels = {str(k): int(v) for k, v in json.load(open(gts[0])).items()} if gts else {}
    img_dir = [d for d in glob.glob(os.path.join(root, '**', 'images'), recursive=True)
               if f'{os.sep}{split}{os.sep}' in d + os.sep][0]
    tracklets = {d: sorted(glob.glob(os.path.join(img_dir, d, '*.jpg'))) for d in sorted(os.listdir(img_dir))}
    return {t: p for t, p in tracklets.items() if p}, labels

def sample_frames(paths, k=6):
    if len(paths) <= k: return list(paths)
    return [paths[i] for i in np.linspace(0, len(paths) - 1, k).round().astype(int)]

train_tr, train_lab = load_split(LOCAL, 'train')
test_tr, test_lab = load_split(LOCAL, 'test')
big_tr, _ = load_split(LOCAL_TRAIN, 'train')

players = sorted(t for t in train_tr if t in train_lab)            # same split as the classical notebook
random.Random(SEED).shuffle(players)
val_ids = set(players[:int(0.2 * len(players))])
dev_ids = [t for t in players if t not in val_ids]

dev_items = [dict(path=p, gt=train_lab[t], tid=t) for t in dev_ids for p in big_tr[t]]          # 16 / player
val_items = [dict(path=p, gt=train_lab[t], tid=t) for t in sorted(val_ids) for p in sample_frames(train_tr[t])]
test_items = [dict(path=p, gt=test_lab[t], tid=t) for t in sorted(test_tr) if t in test_lab
              for p in sample_frames(test_tr[t])]
legible = json.load(open(os.path.join(WORK, 'legible_labels.json')))
print(f'dev {len(dev_items)} frames ({len(dev_ids)} players) | val {len(val_items)} | test {len(test_items)}')

### Torso crops in memory
Same torso crop as the classical pipeline (drop top 1/6, bottom 1/3), resized to 128×96 and kept in RAM so
training is fast.

In [ ]:
IMG_H, IMG_W, MIN_H = 128, 96, 40

def torso_crop(img, top=1/6, bottom=1/3, side=0.08):
    h, w = img.shape[:2]
    y0, y1 = int(round(h * top)), int(round(h * (1 - bottom)))
    x0, x1 = int(round(w * side)), int(round(w * (1 - side)))
    return img[y0:max(y1, y0 + 1), x0:max(x1, x0 + 1)]

def load_crops(items_):
    out = np.zeros((len(items_), IMG_H, IMG_W, 3), np.uint8)
    for i, it in enumerate(items_):
        img = cv2.imread(it['path'])
        it['h'] = img.shape[0] if img is not None else 0
        if img is not None and img.shape[0] >= 8 and img.shape[1] >= 4:
            out[i] = cv2.resize(cv2.cvtColor(torso_crop(img), cv2.COLOR_BGR2RGB), (IMG_W, IMG_H),
                                interpolation=cv2.INTER_CUBIC)
    return out

t0 = time.time()
X_dev, X_val, X_test = load_crops(dev_items), load_crops(val_items), load_crops(test_items)
print(f'loaded crops in {time.time()-t0:.0f}s')

def encode(gt):
    """tens: 0-9 or 10 (= no tens digit) | ones: 0-9 | visible: 0/1"""
    return (10, 0, 0) if gt < 0 else ((gt // 10 if gt >= 10 else 10), gt % 10, 1)

Y_dev = torch.tensor([encode(it['gt']) for it in dev_items])
Y_val = torch.tensor([encode(it['gt']) for it in val_items])
# channel mean/std from OUR data (not ImageNet), on a 1000-image sample to save RAM
_s = X_dev[np.random.RandomState(SEED).choice(len(X_dev), min(1000, len(X_dev)), replace=False)].reshape(-1, 3) / 255.0
MEAN = torch.tensor(_s.mean(0), dtype=torch.float32).view(1, 3, 1, 1).to(DEVICE)
STD = torch.tensor(_s.std(0), dtype=torch.float32).view(1, 3, 1, 1).to(DEVICE)
del _s

### Batches and augmentation (on the GPU)
Images stay as compact uint8 in RAM; each batch is converted to float on the GPU (keeps RAM low).
Random brightness/contrast, small rotation, shift and zoom. **No horizontal flips**: a mirrored digit is a
different (or invalid) digit.

In [ ]:
def to_batch(X, idx):
    """Only the images of one batch become float tensors (on the GPU); the rest stays uint8 in RAM."""
    x = torch.from_numpy(X[idx]).to(DEVICE).permute(0, 3, 1, 2).float().div_(255)
    return (x - MEAN) / STD

def augment(x):
    n = x.shape[0]
    x = x * torch.empty(n, 1, 1, 1, device=x.device).uniform_(0.7, 1.3) \
          + torch.empty(n, 1, 1, 1, device=x.device).uniform_(-0.3, 0.3)            # contrast, brightness
    ang = torch.empty(n, device=x.device).uniform_(-8, 8) * math.pi / 180           # rotation
    sc = torch.empty(n, device=x.device).uniform_(0.9, 1.1)                         # zoom
    tx, ty = (torch.empty(n, device=x.device).uniform_(-.08, .08) for _ in range(2))  # shift
    theta = torch.stack([torch.stack([torch.cos(ang) / sc, -torch.sin(ang) / sc, tx], 1),
                         torch.stack([torch.sin(ang) / sc, torch.cos(ang) / sc, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode='border', align_corners=False)

## The network: designed from scratch
Four convolution blocks (each: two 3×3 convolutions with batch normalization + ReLU, then 2×2 max pooling),
global average pooling, dropout, and three small output layers. About 1.2 million weights, all starting random.

| Block | Output size |
|---|---|
| input | 3 × 128 × 96 |
| block 1 (32 filters) | 32 × 64 × 48 |
| block 2 (64) | 64 × 32 × 24 |
| block 3 (128) | 128 × 16 × 12 |
| block 4 (256) | 256 × 8 × 6 |
| average pool → heads | tens (11), ones (10), visible (2) |

In [ ]:
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.MaxPool2d(2))

class JerseyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.drop = nn.Dropout(0.3)
        self.tens, self.ones, self.vis = nn.Linear(256, 11), nn.Linear(256, 10), nn.Linear(256, 2)
    def forward(self, x):
        f = self.drop(self.features(x).mean((2, 3)))          # global average pooling
        return self.tens(f), self.ones(f), self.vis(f)

def loss_fn(out, y):
    t, o, v = out
    loss = F.cross_entropy(v, y[:, 2])
    m = y[:, 2] == 1                                          # digit losses only where a number exists
    if m.any():
        loss = loss + F.cross_entropy(t[m], y[m, 0], label_smoothing=0.1) \
                    + F.cross_entropy(o[m], y[m, 1], label_smoothing=0.1)
    return loss

model = JerseyNet().to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M weights, all trained from scratch')

## Train on dev players
The checkpoint with the **lowest val loss** is kept (val is for tuning; test is never used here).
Note: labels are per player, so some dev frames are labelled with a number that is not visible in that frame.

In [ ]:
BS = 128
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
steps = EPOCHS * math.ceil(len(X_dev) / BS)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=steps)
hist, best_loss, CKPT = [], 1e9, os.path.join(WORK, 'step7_scratch.pt')

@torch.no_grad()
def val_loss():
    model.eval(); tot = 0.0
    for i in range(0, len(X_val), 256):
        idx = np.arange(i, min(i + 256, len(X_val)))
        xb, yb = to_batch(X_val, idx), Y_val[idx].to(DEVICE)
        tot += loss_fn(model(xb), yb).item() * len(idx)
    return tot / len(X_val)

for ep in range(EPOCHS):
    model.train(); tot, t0 = 0.0, time.time()
    for b in torch.randperm(len(X_dev)).split(BS):
        b = b.numpy()
        xb, yb = augment(to_batch(X_dev, b)), Y_dev[b].to(DEVICE)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(b)
    vl = val_loss(); hist.append((tot / len(X_dev), vl))
    if vl < best_loss:
        best_loss = vl; torch.save(model.state_dict(), CKPT)
    print(f'epoch {ep+1:2d}/{EPOCHS}  train loss {tot/len(X_dev):.3f}  val loss {vl:.3f}  ({time.time()-t0:.0f}s)')

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
plt.figure(figsize=(5, 3)); plt.plot([h[0] for h in hist], label='train'); plt.plot([h[1] for h in hist], label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Training from scratch'); plt.tight_layout(); plt.show()

## Predict every frame
Number = tens·10 + ones (or just ones when "no tens digit"). Confidence = P(visible) × geometric mean of the
digit probabilities, the same idea as the classical reader. A leading 0 or a lone 0 is not a valid number, and
players shorter than 40 px count as unreadable (same rule as the classical pipeline).

In [ ]:
@torch.no_grad()
def predict(X, items_):
    model.eval(); rows = []
    for i in range(0, len(X), 256):
        xb = to_batch(X, np.arange(i, min(i + 256, len(X))))
        t, o, v = (F.softmax(z, 1).cpu().numpy() for z in model(xb))
        for j in range(len(t)):
            it = items_[i + j]
            ti, oi = int(t[j].argmax()), int(o[j].argmax())
            num, dp = (oi, o[j, oi]) if ti == 10 else (ti * 10 + oi, math.sqrt(t[j, ti] * o[j, oi]))
            ok = num > 0 and ti != 0 and it['h'] >= MIN_H
            rows.append(dict(path=it['path'], tid=it['tid'], gt=it['gt'], pred=num if ok else None,
                             conf=float(v[j, 1] * dp) if ok else 0.0, legible=legible.get(it['path'])))
    return rows

rows_val, rows_test = predict(X_val, val_items), predict(X_test, test_items)

## Same metrics as the classical notebook (Step 6)

In [ ]:
def player_metrics(rows, tau):
    by = defaultdict(list)
    for r in rows: by[r['tid']].append(r)
    v = {}
    for tid, rs in by.items():
        votes = Counter()
        for r in rs:
            if r['pred'] is not None and r['conf'] >= tau: votes[r['pred']] += r['conf']
        v[tid] = (votes.most_common(1)[0][0] if votes else -1, rs[0]['gt'])
    vis = [(p, g) for p, g in v.values() if g >= 0]; inv = [(p, g) for p, g in v.values() if g < 0]
    ans = [(p, g) for p, g in vis if p != -1]
    return dict(tau=float(tau), players=len(v),
                overall=sum(p == g for p, g in v.values()) / max(1, len(v)),
                visible_acc=sum(p == g for p, g in vis) / max(1, len(vis)),
                visible_coverage=len(ans) / max(1, len(vis)),
                visible_precision=sum(p == g for p, g in ans) / max(1, len(ans)),
                abstain_on_no_number=sum(p == -1 for p, g in inv) / max(1, len(inv)))

def fmt(d):
    return ' | '.join(f'{k}={v:.2f}' if isinstance(v, float) else f'{k}={v}' for k, v in d.items())

TAUS = np.round(np.arange(0, 0.96, 0.05), 2)
pm_val = [player_metrics(rows_val, t) for t in TAUS]
for m in pm_val[::2]: print('VAL ', fmt(m))
TAU_DEEP = max(pm_val, key=lambda m: m['overall'])['tau']
deep = dict(tau=TAU_DEEP, val=player_metrics(rows_val, TAU_DEEP), test=player_metrics(rows_test, TAU_DEEP))
print(f'\nchosen τ = {TAU_DEEP} (on val)\nTEST:', fmt(deep['test']))
json.dump(deep, open(os.path.join(WORK, 'step7_scratch_results.json'), 'w'), indent=2)

## Classical vs deep (all trained from scratch): the comparison table
Reads the classical results saved by the main notebook (`step6_results.json`, `partB_results.json`).

In [ ]:
s6 = json.load(open(os.path.join(WORK, 'step6_results.json')))
cls = s6['results'][s6['main']]
systems = {f"Classical ({s6['main']})": cls['test']}
if os.path.exists(os.path.join(WORK, 'partB_results.json')):
    pb = json.load(open(os.path.join(WORK, 'partB_results.json')))
    systems['Classical + visibility gate'] = pb['gate']['test']
systems['Deep CNN from scratch'] = deep['test']

keys = [('overall', 'Overall (SoccerNet-style)'), ('visible_acc', 'Visible: read'),
        ('visible_coverage', 'Visible: answered'), ('visible_precision', 'Correct when answering'),
        ('abstain_on_no_number', 'No number: "unreadable"')]
print(f"{'TEST players (n=1211)':28s}" + ''.join(f'{n[:22]:>24s}' for n in systems))
for k, name in keys:
    print(f'{name:28s}' + ''.join(f'{s[k]:24.2f}' for s in systems.values()))
base_label = 'always "unreadable"'
print(f"{base_label:28s}{s6['always_unreadable']:24.2f}")
print(f"\nval → test overall: classical {cls['val']['overall']:.2f} → {cls['test']['overall']:.2f} | "
      f"deep {deep['val']['overall']:.2f} → {deep['test']['overall']:.2f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
xs, w = np.arange(len(keys)), 0.8 / len(systems)
for i, (n, s) in enumerate(systems.items()):
    ax.bar(xs + (i - (len(systems) - 1) / 2) * w, [s[k] for k, _ in keys], w, label=n)
ax.axhline(s6['always_unreadable'], ls='--', c='gray', lw=1, label='always "unreadable"')
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in keys], rotation=20, ha='right', fontsize=8)
ax.set_ylim(0, 1); ax.set_title('Test players: classical pipeline vs deep CNN (both trained from scratch)')
ax.legend(fontsize=7, loc='upper left')
plt.tight_layout(); plt.savefig(os.path.join(WORK, 'figures', 'classical_vs_deep.png'), dpi=150); plt.show()